# Create NSF Sri Lanka Awards from GMIS Research & Technology Grants

Creates OpenAlex award rows from the National Science Foundation (NSF) of Sri Lanka's official Grant Management Information System (GMIS) Research & Technology Grant detail pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/nsf_sri_lanka_to_s3.py` to enumerate the official public GMIS detail pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://gmis.nsf.gov.lk/` and `https://gmis.nsf.gov.lk/rtgrainfodetailView/{detail_id}`  
**Dashboard corroboration:** `https://gmis.nsf.gov.lk/homedashboard` reports 2,098 research grants plus 81 technology grants.  
**S3 location:** `s3a://openalex-ingest/awards/nsf_sri_lanka/nsf_sri_lanka_grants.parquet`  
**Local full-source validation on 2026-05-27:** 2,167 public GMIS grant detail pages enumerated from IDs 1-2,300; dashboard reports 2,098 research grants plus 81 technology grants; 100% grant-number/title/landing-url coverage, 99.7% principal-investigator coverage, 99.3% inferred source-year coverage, and 90.9% subject coverage; amount/currency intentionally NULL because GMIS does not publish per-grant money.

**NSF Sri Lanka funder:**
- funder_id: 4320335353
- display_name: `National Science Foundation of Sri Lanka`
- country: LK

**Amount/currency decision:** GMIS public grant detail pages do not publish per-grant amounts or currencies. Amount and currency are intentionally NULL under the runbook's amount-waiver rule; the notebook preserves grant numbers, subjects, keywords, investigators, and source URLs for audit.

**Mapping summary:** One OpenAlex award row per GMIS Research & Technology Grant detail page. Native key is `nsf-sri-lanka-{grant_number}` normalized from the official grant number.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nsf_sri_lanka_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/nsf_sri_lanka/nsf_sri_lanka_grants.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 2,167 rows.
SELECT COUNT(*) as total_nsf_sri_lanka_grants
FROM openalex.awards.nsf_sri_lanka_awards_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.nsf_sri_lanka_awards_raw;

In [ ]:
%sql
-- Sample raw NSF Sri Lanka data.
SELECT
    detail_id,
    funder_award_id,
    grant_number,
    display_name,
    principal_investigators,
    subject,
    keywords,
    source_year,
    landing_page_url
FROM openalex.awards.nsf_sri_lanka_awards_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Dates, Native Keys, and Amount Waiver

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'nsf_sri_lanka_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
-- Amounts/currencies are expected to be NULL because GMIS does not publish per-grant money.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year,
    COUNT(start_date) AS rows_with_start_date,
    COUNT(end_date) AS rows_with_end_date
FROM openalex.awards.nsf_sri_lanka_awards_raw;

In [ ]:
%sql
-- Native-key inspection. `funder_award_id` is derived from the official NSF grant number.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT detail_id) AS distinct_detail_ids,
    COUNT(DISTINCT grant_number) AS distinct_grant_numbers,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.nsf_sri_lanka_awards_raw;

In [ ]:
%sql
-- Source subject and year distributions.
SELECT subject, COUNT(*) AS cnt
FROM openalex.awards.nsf_sri_lanka_awards_raw
GROUP BY subject
ORDER BY cnt DESC
LIMIT 50;

In [ ]:
%sql
SELECT TRY_CAST(source_year AS INT) AS source_year, COUNT(*) AS cnt
FROM openalex.awards.nsf_sri_lanka_awards_raw
GROUP BY TRY_CAST(source_year AS INT)
ORDER BY source_year DESC;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320335353;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.nsf_sri_lanka_awards
USING delta
AS
WITH
nsf_sri_lanka_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320335353
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN currency ELSE NULL END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year,
        CONCAT_WS(
            ' | ',
            NULLIF(TRIM(abstract), ''),
            NULLIF(TRIM(key_research_findings), '')
        ) AS source_description
    FROM openalex.awards.nsf_sri_lanka_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN NULLIF(TRIM(g.source_description), '') IS NOT NULL THEN NULLIF(TRIM(g.source_description), '')
            WHEN NULLIF(TRIM(g.keywords), '') IS NOT NULL THEN CONCAT('Keywords: ', TRIM(g.keywords))
            ELSE NULL
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        NULLIF(TRIM(g.subject), '') as funder_scheme,

        'nsf_sri_lanka_gmis' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,

        struct(
            NULLIF(TRIM(g.lead_investigator_given_name), '') as given_name,
            NULLIF(TRIM(g.lead_investigator_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                CAST(NULL AS STRING) as name,
                'LK' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN nsf_sri_lanka_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 138

In [ ]:
%sql
-- Remove previous NSF Sri Lanka GMIS data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nsf_sri_lanka_gmis' AND priority = 138;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    138 as priority
FROM openalex.awards.nsf_sri_lanka_awards;

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_nsf_sri_lanka_awards
FROM openalex.awards.nsf_sri_lanka_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.nsf_sri_lanka_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS lead_given_name,
    lead_investigator.family_name AS lead_family_name,
    landing_page_url
FROM openalex.awards.nsf_sri_lanka_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.nsf_sri_lanka_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only NSF Sri Lanka.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.nsf_sri_lanka_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(funder_scheme) as has_subject,
    COUNT(lead_investigator.family_name) as has_lead_family_name,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(funder_scheme), COUNT(*)) * 100.0, 1) as pct_subject
FROM openalex.awards.nsf_sri_lanka_awards;

In [ ]:
%sql
-- Amount/currency waiver check: public GMIS detail pages do not publish per-grant amounts.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.nsf_sri_lanka_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.nsf_sri_lanka_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Subject distribution.
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.nsf_sri_lanka_awards
GROUP BY funder_scheme
ORDER BY cnt DESC
LIMIT 50;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 138.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'nsf_sri_lanka_gmis' AND priority = 138
GROUP BY priority, provenance;

## Handoff Notes

Contractor has no S3/Databricks access. Admin must:
- Run `scripts/local/nsf_sri_lanka_to_s3.py --allow-shrink` or equivalent with AWS credentials to upload the parquet.
- Run this notebook on Databricks.
- Run the Step 6 verification cells.
- Mark the tracker Complete only after Databricks QA and production/API verification.
